In [ ]:
from strands import Agent, tool
from duckduckgo_search import DDGS
from duckduckgo_search.exceptions import (
    RatelimitException, 
    DuckDuckGoSearchException,
)
import logging

In [ ]:
# 로깅 구성
logging.getLogger("strands").setLevel(logging.INFO)

In [ ]:
# 웹 검색 도구 정의
@tool
def websearch(keywords: str, region: str = "us-en", max_results: int | None = None) -> str:
    """웹을 검색하여 최신 정보를 얻습니다.
    Args:
        keywords (str): 검색 쿼리 키워드
        region (str): 검색 지역: us-en, uk-en, ru-ru 등
        max_results (int | None): 반환할 최대 결과 수
    Returns:
        검색 결과가 포함된 딕셔너리 목록
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "검색 결과가 없습니다."
    except RatelimitException:
        return "RatelimitException: 나중에 다시 시도하세요."
    except DuckDuckGoSearchException as d:
        return f"DuckDuckGo 검색 오류: {d}"
    except Exception as e:
        return f"Exception: {e}"

In [ ]:
# 레시피 도우미 에이전트 생성
recipe_agent = Agent(
    system_prompt="""당신은 RecipeBot, 도움이 되는 요리 도우미입니다.
    사용자가 재료를 언급하면 레시피를 찾고 요리 관련 질문에 답변하세요.
    사용자가 재료를 언급하거나 요리 정보를 찾을 때 웹 검색 도구를 사용하세요.""",
    tools=[websearch], # 위에서 만든 웹 검색 도구 가져오기
)

In [ ]:
response = recipe_agent("닭고기와 브로콜리를 사용한 레시피를 제안해주세요.")

print(f"Metrics : {response.metrics}") # 선택 사항이지만 권장됩니다.